# 在 Colab 訓練三類食物模型（全穀雜糧／蔬菜／豆魚蛋肉）

**事前準備**：在 Mac 上執行 `bash training/pack_for_colab.sh`，把產生的 `smart-health-diary-colab.zip` 上傳到 Google 雲端硬碟的「我的雲端硬碟」根目錄。

**開始前**：選單「執行階段 → 變更執行階段類型」選擇 **T4 GPU**，然後由上往下逐格執行。

In [ ]:
# 1. 確認有 GPU（應該看到 Tesla T4）
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. 連接 Google 雲端硬碟並解壓縮專案
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/smart-health-diary && mkdir -p /content/smart-health-diary
!unzip -q /content/drive/MyDrive/smart-health-diary-colab.zip -d /content/smart-health-diary
%cd /content/smart-health-diary
!ls training/foodgroups

In [ ]:
# 3. 安裝套件、下載 gelan-s 預訓練權重
!pip install -q -r backend/requirements-train.txt
!mkdir -p weights && curl -sL --fail -o weights/gelan-s.pt https://github.com/WongKinYiu/yolov9/releases/download/v0.1/gelan-s.pt && ls -lh weights

## 訓練
T4 上 100 個 epoch 大約需要 30–60 分鐘。訓練期間不要關閉分頁。

In [ ]:
%cd /content/smart-health-diary/backend/yolov9
!python train.py \
  --cfg models/detect/gelan-s.yaml \
  --weights ../../weights/gelan-s.pt \
  --data ../../training/foodgroups/foodgroups.yaml \
  --hyp data/hyps/hyp.scratch-high.yaml \
  --img 640 --batch 32 --epochs 100 \
  --device 0 --workers 4 --close-mosaic 15 \
  --name foodgroups_gelan_s --exist-ok

## 評估
先評估單一菜色測試集；如果整盤測試集已經補標完成並改名為 `test_tray`，也會一起評估。

In [ ]:
%cd /content/smart-health-diary
!bash training/evaluate.sh backend/yolov9/runs/train/foodgroups_gelan_s/weights/best.pt 0

In [ ]:
# 查看訓練曲線與混淆矩陣
from IPython.display import Image, display
run = '/content/smart-health-diary/backend/yolov9/runs/train/foodgroups_gelan_s'
display(Image(f'{run}/results.png', width=900))
display(Image(f'{run}/confusion_matrix.png', width=600))

## 存回雲端硬碟
Colab 關閉後檔案會消失，記得把結果存回雲端硬碟。

In [ ]:
!mkdir -p /content/drive/MyDrive/foodgroups_results
!cp -r /content/smart-health-diary/backend/yolov9/runs/train/foodgroups_gelan_s /content/drive/MyDrive/foodgroups_results/
!cp -r /content/smart-health-diary/backend/yolov9/runs/val /content/drive/MyDrive/foodgroups_results/ 2>/dev/null || true
!ls /content/drive/MyDrive/foodgroups_results/foodgroups_gelan_s/weights